# 5ifthNN — Smart Incremental Colab L4 Training & Versioned ONNX Export

**Just click `Runtime → Run All`:**
- Supports **both** `.zip` bundles and loose `.5nn.gz` files mixed together in `MyDrive/5ifthNN/recordings/`.
- **Smart Session & Crash-Safe Deduplication:** Tracks every trained dataset in `checkpoints/training_state.json` with a unique `SESSION_ID`. If you re-run with no new data, it skips already-trained datasets automatically. If a session crashes mid-run, nothing is marked trained so a new session picks up cleanly.
- **Automatic Versioning:** Exports `fifthnn_v1_bc_sXXXXXX.onnx`, `fifthnn_v2_ppo_sYYYYYY.onnx`, etc. into `MyDrive/5ifthNN/models/` (plus `fifthnn_latest.onnx`) so you always know which model is newest.

In [ ]:
# 1. Mount Google Drive, sync 5ifthNN repo & install dependencies
from google.colab import drive
drive.mount('/content/drive')

!nvidia-smi
!git clone https://github.com/5ivesaw/5ifthNN.git /content/5ifthNN || (cd /content/5ifthNN && git pull)
%cd /content/5ifthNN/python
!pip -q install -r requirements.txt

In [ ]:
# 2. Initialize Session ID, load training_state.json, unpack ALL zips + loose .5nn.gz files & Audit
import glob, json, os, random, shutil, zipfile
from pathlib import Path

SESSION_ID = f's{random.randint(100000, 999999)}'
DATA_DIR = '/content/drive/MyDrive/5ifthNN/recordings'
ROLLOUT_DIR = '/content/drive/MyDrive/5ifthNN/selfplay_rollouts'
CKPT_DIR = '/content/drive/MyDrive/5ifthNN/checkpoints'
MODEL_DIR = '/content/drive/MyDrive/5ifthNN/models'
UNPACK_BC = '/content/fifthnn_unpack_bc'
UNPACK_PPO = '/content/fifthnn_unpack_ppo'
STATE_FILE = Path(CKPT_DIR) / 'training_state.json'

for d in (DATA_DIR, ROLLOUT_DIR, CKPT_DIR, MODEL_DIR, UNPACK_BC, UNPACK_PPO):
    os.makedirs(d, exist_ok=True)

state = json.loads(STATE_FILE.read_text()) if STATE_FILE.exists() else {
    'model_version': 0,
    'trained_bc_files': [],
    'trained_ppo_files': [],
    'history': [],
}
print(f'Session: {SESSION_ID} | Current Model Version: v{state["model_version"]}')

def unpack_folder(src_dir: str, dst_dir: str) -> list[str]:
    # Unpack any .zip archives (combined or individual) + symlink loose .5nn.gz files
    for zf in sorted(glob.glob(f'{src_dir}/**/*.zip', recursive=True)):
        with zipfile.ZipFile(zf, 'r') as z:
            for member in z.namelist():
                if member.endswith(('.5nn.gz', '.5nn', '.meta.json', '.tags.jsonl')):
                    out = Path(dst_dir) / Path(member).name
                    if not out.exists():
                        out.write_bytes(z.read(member))
    for f in sorted(glob.glob(f'{src_dir}/**/*.5nn*', recursive=True)):
        dst = Path(dst_dir) / Path(f).name
        if not dst.exists():
            os.symlink(f, dst)
    return sorted(Path(p).name for p in glob.glob(f'{dst_dir}/*.5nn*') if p.endswith(('.5nn', '.5nn.gz')))

all_bc_files = unpack_folder(DATA_DIR, UNPACK_BC)
all_ppo_files = unpack_folder(ROLLOUT_DIR, UNPACK_PPO)
new_bc_files = [f for f in all_bc_files if f not in set(state['trained_bc_files'])]
new_ppo_files = [f for f in all_ppo_files if f not in set(state['trained_ppo_files'])]

print(f'BC recordings:  {len(all_bc_files)} total | {len(new_bc_files)} NEW')
print(f'PPO rollouts:   {len(all_ppo_files)} total | {len(new_ppo_files)} NEW')
if all_bc_files:
    !python3 -m fifthnn.audit "$UNPACK_BC"

In [ ]:
# 3. Phase 2A: Smart Behavior Cloning (Skips if no new BC files; resumes if previous BC exists!)
import subprocess

BC_CKPT = f'{CKPT_DIR}/fifthnn_bc_latest.pt'
trained_stage = None

if new_bc_files:
    cmd = [
        'python3', '-m', 'fifthnn.train_bc',
        '--data_dir', UNPACK_BC,
        '--out', BC_CKPT,
        '--epochs', '25',
        '--batch_size', '32',
        '--seq_len', '128',
        '--lr', '3e-4',
    ]
    if os.path.exists(BC_CKPT):
        cmd += ['--resume', BC_CKPT]
        print(f'[{SESSION_ID}] Fine-tuning BC checkpoint on {len(new_bc_files)} new file(s)...')
    else:
        print(f'[{SESSION_ID}] Training fresh BC starter brain on {len(all_bc_files)} file(s)...')
    subprocess.run(cmd, check=True)
    state['trained_bc_files'] = sorted(set(state['trained_bc_files']) | set(all_bc_files))
    trained_stage = 'bc'
else:
    print(f'[{SESSION_ID}] No new BC recordings -> skipping BC (already trained on all {len(all_bc_files)} files).')

In [ ]:
# 4. Phase 2B: Smart KL-Anchored PPO (Runs ONLY on new AI rollouts in selfplay_rollouts/!)
PPO_CKPT = f'{CKPT_DIR}/fifthnn_ppo_latest.pt'

if new_ppo_files and os.path.exists(BC_CKPT):
    # Stage only NEW rollouts so PPO never re-trains on old rollouts
    PPO_NEW_DIR = f'/content/fifthnn_ppo_new_{SESSION_ID}'
    os.makedirs(PPO_NEW_DIR, exist_ok=True)
    for fname in new_ppo_files:
        for suffix in ('', '.meta.json', '.tags.jsonl'):
            src = Path(UNPACK_PPO) / (fname + suffix)
            if src.exists():
                os.symlink(src, Path(PPO_NEW_DIR) / src.name)
    print(f'[{SESSION_ID}] Running KL-PPO on {len(new_ppo_files)} NEW rollout file(s)...')
    subprocess.run([
        'python3', '-m', 'fifthnn.train_ppo',
        '--rollout_dir', PPO_NEW_DIR,
        '--bc_anchor', BC_CKPT,
        '--policy', PPO_CKPT if os.path.exists(PPO_CKPT) else BC_CKPT,
        '--out', PPO_CKPT,
        '--kl_coef', '0.08',
        '--epochs', '4',
    ], check=True)
    state['trained_ppo_files'] = sorted(set(state['trained_ppo_files']) | set(new_ppo_files))
    trained_stage = 'ppo'
else:
    print(f'[{SESSION_ID}] No new PPO rollouts -> skipping PPO.')

In [ ]:
# 5. Versioned ONNX Export (v1, v2, v3... + fifthnn_latest.onnx) & Save State
if trained_stage is not None:
    state['model_version'] += 1
    ver = state['model_version']
    export_ckpt = PPO_CKPT if trained_stage == 'ppo' else BC_CKPT
    ver_onnx = f'{MODEL_DIR}/fifthnn_v{ver}_{trained_stage}_{SESSION_ID}.onnx'
    latest_onnx = f'{MODEL_DIR}/fifthnn_latest.onnx'

    subprocess.run([
        'python3', '-m', 'fifthnn.export_onnx',
        '--ckpt', export_ckpt,
        '--out', ver_onnx,
        '--quantize_int8',
    ], check=True)
    shutil.copy2(ver_onnx, latest_onnx)

    state['history'].append({
        'version': ver,
        'stage': trained_stage,
        'session': SESSION_ID,
        'onnx': Path(ver_onnx).name,
        'bc_files_total': len(state['trained_bc_files']),
        'ppo_files_total': len(state['trained_ppo_files']),
    })
    STATE_FILE.write_text(json.dumps(state, indent=2))
    print(f'\n=== EXPORTED v{ver} ({trained_stage.upper()}) ===')
    print(f'Versioned: {ver_onnx}')
    print(f'Latest:    {latest_onnx}')
else:
    print(f'[{SESSION_ID}] Nothing new to train or export! Latest model is still v{state["model_version"]}.')